# 05 — MCP SSE Transport

Connects to a running MCP server over **HTTP/SSE** instead of launching a subprocess.

Use the SSE transport when:
- The MCP server runs independently (e.g. in Docker)
- Multiple clients share the same server instance
- The server is remote / behind authentication

**Prerequisites**: An MCP server listening at `http://localhost:8000/sse`  
(start with e.g. `npx @modelcontextprotocol/server-everything`).

In [1]:
import asyncio
from agent_framework.extensions.mcp import MCPClient, MCPTool
from agent_framework.providers.llm.openai.openai_client import OpenAIClient
from agent_framework.core.memory.unbounded_memory import UnboundedMemory
from agent_framework.core.messages.client_messages import UserMessage, SystemMessage

## Connect via SSE transport

In [6]:
import json
from agent_framework.core.messages.client_messages import ToolExecutionResultMessage

SSE_URL = 'http://localhost:9000/sse'  # MCP server: docker compose --profile mcp up -d mcp-server

async def demo_sse():
    mcp_client = MCPClient()
    try:
        await mcp_client.connect_sse(url=SSE_URL, headers={}, timeout=30.0)
        print(f'Connected via {mcp_client.transport_type} transport')

        mcp_tools = await MCPTool.from_mcp_client(mcp_client)
        print(f'Discovered {len(mcp_tools)} tools:')
        for t in mcp_tools:
            print(f'  - {t.name}: {t.description}')

        tool_map = {t.name: t for t in mcp_tools}
        client = OpenAIClient(model='gpt-4o')
        memory = UnboundedMemory()
        memory.add_message(SystemMessage(content='You are a helpful assistant with tools via MCP.'))
        memory.add_message(UserMessage(content=[{'type': 'text', 'text': 'Add 42 and 58, then echo back "MCP SSE works!"'}]))

        # First LLM call — returns tool calls
        response = await client.generate(
            messages=memory.get_messages(),
            tools=[t.get_openai_schema() for t in mcp_tools],
        )

        if response.tool_calls:
            print(f'\nLLM requested {len(response.tool_calls)} tool call(s):')
            memory.add_message(response)  # add assistant message with tool_calls
            for tc in response.tool_calls:
                # ToolCallMessage fields: id, name, arguments (already a dict or JSON string)
                name = tc.name
                args = tc.arguments if isinstance(tc.arguments, dict) else json.loads(tc.arguments)
                print(f'  → {name}({args})')
                tool = tool_map.get(name)
                if tool:
                    result = await tool.execute(**args)
                    result_text = result.content[0]['text'] if result.content else ''
                    print(f'  ← {result_text}')
                    memory.add_message(ToolExecutionResultMessage(
                        content=result_text,
                        tool_call_id=tc.id,
                        name=name,
                    ))

            # Second LLM call — get final text response
            final = await client.generate(messages=memory.get_messages(), tools=[])
            print(f'\nFinal response: {final.content}')
        else:
            print(f'\nDirect response: {response.content}')

    except (RuntimeError, OSError, ConnectionRefusedError) as e:
        print(f'⚠ Connection error: {e}')
        print(f'  Start the MCP server: docker compose --profile mcp up -d mcp-server')
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'Error: {type(e).__name__}: {e}')
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()
            print('Disconnected.')

await demo_sse()


Connected via sse transport
Discovered 7 tools:
  - add: Add two numbers.
  - subtract: Subtract b from a.
  - multiply: Multiply two numbers.
  - echo: Echo a message back.
  - to_uppercase: Convert text to uppercase.
  - word_count: Count words and characters in text.
  - server_info: Return metadata about this MCP server.

LLM requested 2 tool call(s):
  → add({'a': 42, 'b': 58})
  ← 100.0
  → echo({'message': 'MCP SSE works!'})
  ← MCP SSE works!

Final response: ['The sum of 42 and 58 is 100. MCP SSE works!']
Disconnected.


## Stdio vs SSE — side-by-side comparison

In [7]:
print('Stdio transport:')
print('  + Launches server as subprocess')
print('  + Automatic lifecycle management')
print('  + Good for local development')
print('  - One client per server instance')
print()
print('SSE transport:')
print('  + Connects to a running server')
print('  + Multiple clients can share one server')
print('  + Supports authentication headers')
print('  + Good for production / remote servers')

Stdio transport:
  + Launches server as subprocess
  + Automatic lifecycle management
  + Good for local development
  - One client per server instance

SSE transport:
  + Connects to a running server
  + Multiple clients can share one server
  + Supports authentication headers
  + Good for production / remote servers
